# Publishing & Searching Interactions

> **Time:** ~12 minutes | **Level:** Beginner

**Interactions** are the raw data in Reflexio — every message exchanged between a user and your agent.
This notebook covers the three interaction types and how to search them:

- **Text conversations** — plain user/agent message pairs
- **Tool-augmented interactions** — messages that record which tools the agent called
- **User actions** — non-text events like clicks and scrolls

We will use a fictional **Acme Electronics** support agent throughout.

In [ ]:
import time
import uuid

from _display_helpers import load_env, show_interactions, show_success

from reflexio import InteractionData, ReflexioClient, ToolUsed, UserActionType
from reflexio.models.config_schema import SearchMode

url, api_key = load_env()
client = ReflexioClient(url_endpoint=url, api_key=api_key)

# Unique run ID so multiple notebook runs don't collide
RUN_ID = uuid.uuid4().hex[:8]
USER_ID = f"acme_customer_{RUN_ID}"
print(f"Run ID: {RUN_ID}")
print(f"User ID: {USER_ID}")

## Text Conversations

The simplest interaction type: a multi-turn text exchange between a customer and the agent.
Each turn is an `InteractionData` with a **role** (`"User"` or `"Agent"`) and **content**.

In [ ]:
response = client.publish_interaction(
    user_id=USER_ID,
    interactions=[
        InteractionData(
            role="User",
            content="Hi, I just noticed a charge of $49.99 on my credit card from Acme Electronics. I don't remember making this purchase.",
            user_action=UserActionType.NONE,
        ),
        InteractionData(
            role="Agent",
            content="Let me look into that for you. I can see order ORD-2847 placed on March 15 for an Acme Wireless Mouse. Does that ring a bell?",
            user_action=UserActionType.NONE,
        ),
        InteractionData(
            role="User",
            content="Oh wait, yes — my son must have ordered it from my account. Sorry about the confusion!",
            user_action=UserActionType.NONE,
        ),
        InteractionData(
            role="Agent",
            content="No worries at all! The order is currently being shipped and should arrive by Thursday. Would you like me to set up a separate account for your son?",
            user_action=UserActionType.NONE,
        ),
        InteractionData(
            role="User",
            content="That would be great, actually. His name is Jake. Also, can you make sure all my order confirmations go to my work email?",
            user_action=UserActionType.NONE,
        ),
        InteractionData(
            role="Agent",
            content="Done! I've created a linked account for Jake and updated your notification email to your work address. Is there anything else I can help with?",
            user_action=UserActionType.NONE,
        ),
    ],
    source="notebook",
    session_id=f"billing_{RUN_ID}",
    wait_for_response=True,
)
show_success("Published billing conversation (6 turns)")

In [ ]:
response = client.publish_interaction(
    user_id=USER_ID,
    interactions=[
        InteractionData(
            role="User",
            content="My Acme ProBook keyboard stopped working after just 8 months. Is this covered under warranty?",
            user_action=UserActionType.NONE,
        ),
        InteractionData(
            role="Agent",
            content="Yes, your ProBook comes with a 1-year standard warranty. Since it's only 8 months old, the keyboard replacement is fully covered.",
            user_action=UserActionType.NONE,
        ),
        InteractionData(
            role="User",
            content="Great. How do I get it fixed? I need the laptop for work so I can't be without it for long.",
            user_action=UserActionType.NONE,
        ),
        InteractionData(
            role="Agent",
            content="We have two options: mail-in repair (3-5 business days) or same-day service at an Acme Store. The downtown location has availability tomorrow at 10 AM.",
            user_action=UserActionType.NONE,
        ),
        InteractionData(
            role="User",
            content="Same-day at the downtown store sounds perfect. Can you book the 10 AM slot for me?",
            user_action=UserActionType.NONE,
        ),
        InteractionData(
            role="Agent",
            content="Booked! You're confirmed for tomorrow at 10 AM at the Downtown Acme Store. Bring your laptop and photo ID. The repair usually takes about 45 minutes.",
            user_action=UserActionType.NONE,
        ),
    ],
    source="notebook",
    session_id=f"warranty_{RUN_ID}",
    wait_for_response=True,
)
show_success("Published warranty conversation (6 turns)")

## Tool-Augmented Interactions

When your agent calls a tool (e.g., an order lookup API), you can record it with a `ToolUsed` object attached to the agent's turn. It has two fields:

- **`tool_name`** — the name of the tool that was called
- **`tool_data`** — a dict of tool metadata: input arguments, output/result, latency, or any other info

In [ ]:
resp = client.publish_interaction(
    user_id=USER_ID,
    interactions=[
        InteractionData(
            role="User",
            content="Where is my order ORD-5521? It was supposed to arrive yesterday.",
        ),
        InteractionData(
            role="Agent",
            content="Let me look that up for you. Your order ORD-5521 has shipped and is currently in transit. The estimated delivery is April 5.",
            tools_used=[
                ToolUsed(
                    tool_name="order_lookup",
                    tool_data={"input": {"order_id": "ORD-5521"}, "output": "Status: Shipped, ETA: April 5"},
                )
            ],
        ),
    ],
    source="web_chat",
    agent_version="v1.0",
    wait_for_response=True,
)
show_success(f"Published tool-augmented interaction: {resp.success}")

In [ ]:
# Retrieve and see how tool usage is stored
from rich import print as rprint

resp = client.get_interactions(user_id=USER_ID, top_k=5)
tool_interactions = [i for i in resp.interactions if i.tools_used]
show_interactions(tool_interactions, title="Interactions with Tool Usage")

# Show tool details
for interaction in tool_interactions:
    for t in interaction.tools_used:
        rprint(f"  Tool: [bold]{t.tool_name}[/bold]")
        rprint(f"  Input: {t.tool_data.get("input", "")}")
        rprint(f"  Result: {t.tool_data.get("output", "")}")

## User Actions (Clicks, Scrolls)

Not every interaction is a message. Users also click buttons, scroll pages, and navigate.
Use `user_action` and `user_action_description` to capture these events.

In [ ]:
resp = client.publish_interaction(
    user_id=USER_ID,
    interactions=[
        InteractionData(
            role="User",
            content="I want to see where my package is right now.",
            user_action=UserActionType.CLICK,
            user_action_description="Clicked 'Track My Order' button",
        ),
        InteractionData(
            role="Agent",
            content="Your package is currently at the regional distribution center in Dallas, TX. It should arrive by tomorrow evening.",
        ),
    ],
    source="mobile_app",
    agent_version="v1.0",
    wait_for_response=True,
)
show_success(f"Published user action interaction: {resp.success}")

In [ ]:
# Retrieve and see how user actions are stored
resp = client.search_interactions(user_id=USER_ID, query="track my order", top_k=3)
for interaction in resp.interactions:
    if interaction.user_action and str(interaction.user_action) != "none":
        rprint(f"  Action: [bold]{interaction.user_action}[/bold]")
        rprint(f"  Description: {interaction.user_action_description}")
        rprint(f"  Content: {interaction.content}")

## Session Grouping

> Group related interactions with `session_id` to track multi-step conversations.

In [ ]:
SESSION = f"session_{RUN_ID}"

# First request in the session: customer asks about return policy
client.publish_interaction(
    user_id=USER_ID,
    session_id=SESSION,
    interactions=[
        InteractionData(role="User", content="What is your return policy for opened electronics?"),
        InteractionData(role="Agent", content="Acme Electronics offers a 30-day return policy for opened items, as long as all original accessories are included."),
    ],
    source="web_chat",
    agent_version="v1.0",
    wait_for_response=True,
)

# Second request in the same session: follow-up question
client.publish_interaction(
    user_id=USER_ID,
    session_id=SESSION,
    interactions=[
        InteractionData(role="User", content="Great. I'd like to return the wireless earbuds I bought last week. They don't fit well."),
        InteractionData(role="Agent", content="I can help with that. I've initiated a return for your wireless earbuds (order ORD-8834). You'll receive a prepaid shipping label via email within the hour."),
    ],
    source="web_chat",
    agent_version="v1.0",
    wait_for_response=True,
)

show_success(f"Published 2 requests under session '{SESSION}'")

In [ ]:
# Retrieve requests and show session grouping
resp = client.get_requests(user_id=USER_ID, top_k=20)
for session in resp.sessions:
    rprint(f"\n[bold]Session: {session.session_id}[/bold] ({len(session.requests)} request(s))")
    for req_data in session.requests:
        turns = len(req_data.interactions)
        rprint(f"  Request {req_data.request.request_id[:12]}... — {turns} turns, source={req_data.request.source}")

## Fire-and-Forget vs Synchronous

- **`wait_for_response=False`** (default) — returns `None` immediately; the server processes profile extraction and playbook generation in the background. Your agent doesn't wait, so the user sees a response faster.
- **`wait_for_response=True`** — blocks until the server finishes all processing (extraction, embedding, storage) and returns the result.

Use **fire-and-forget** in production to keep your agent's response time low — the publish call returns in milliseconds instead of waiting seconds for extraction to complete. Use **synchronous** mode in notebooks and tests where you need to verify results immediately after publishing.

In [ ]:
# Fire-and-forget: returns None immediately
t0 = time.time()
result_async = client.publish_interaction(
    user_id=USER_ID,
    interactions=[
        InteractionData(role="User", content="Do you sell laptop stands?"),
        InteractionData(role="Agent", content="Yes! We have several aluminum laptop stands starting at $39.99."),
    ],
    source="web_chat",
    agent_version="v1.0",
    wait_for_response=False,
)
t_async = time.time() - t0
print(f"Fire-and-forget: returned {result_async} in {t_async:.3f}s")

# Synchronous: blocks until processed
t0 = time.time()
result_sync = client.publish_interaction(
    user_id=USER_ID,
    interactions=[
        InteractionData(role="User", content="What colors do the laptop stands come in?"),
        InteractionData(role="Agent", content="We offer them in silver, space gray, and rose gold."),
    ],
    source="web_chat",
    agent_version="v1.0",
    wait_for_response=True,
)
t_sync = time.time() - t0
print(f"Synchronous:     returned success={result_sync.success} in {t_sync:.3f}s")

## List Recent Interactions

Retrieve the most recent interactions for a user without any search query.

In [ ]:
resp = client.get_interactions(user_id=USER_ID, top_k=10)
show_interactions(resp.interactions, title=f"Recent Interactions for {USER_ID}")

### Most Recent K

Use `most_recent_k` in `search_interactions` to return only the last *k* interactions,
sorted by recency. This is handy for building a context window for your agent.

In [ ]:
resp = client.get_interactions(user_id=USER_ID, top_k=4)
show_interactions(resp.interactions, title="Most Recent 4 Interactions")

## Search Interactions

Use semantic search to find interactions matching a natural-language query.

In [ ]:
resp = client.search_interactions(
    user_id=USER_ID,
    query="billing charge",
    top_k=5,
)
show_interactions(resp.interactions, title="Search: 'billing charge'")

## Search Modes: Vector, FTS, Hybrid

> **Vector** = semantic meaning (finds related concepts even without keyword overlap).  
> **FTS** = keyword matching (fast, exact-term lookup).  
> **Hybrid** = best of both, using Reciprocal Rank Fusion (default).

In [ ]:

QUERY = "order delivery status"

for mode in [SearchMode.VECTOR, SearchMode.FTS, SearchMode.HYBRID]:
    resp = client.search_interactions(
        user_id=USER_ID,
        query=QUERY,
        top_k=3,
        search_mode=mode,
    )
    show_interactions(
        resp.interactions,
        title=f"{mode.value.upper()} results for '{QUERY}'",
    )

In [ ]:
# --- Cleanup: remove all data created by this notebook ---
client.delete_all_interactions()
client.delete_all_profiles()
show_success("Cleanup complete.")

## Summary & Next Steps

In this notebook you learned how to:

- **Publish** text, tool-augmented, and user-action interactions
- **Group** interactions into sessions with `session_id`
- Choose between **fire-and-forget** and **synchronous** publishing
- **List** and **search** interactions using vector, FTS, and hybrid modes

Next, head to **`02_profiles.ipynb`** to learn how Reflexio automatically extracts and manages user profiles from these interactions.